## Importar librerías y cargar API key de OpenRouter

In [ ]:
!pip install openai

In [ ]:
# Importación de librerías necesarias
import pandas as pd
import re
import tiktoken
import time
import os
from openai import OpenAI
from google.genai import types
from google.colab import userdata
from transformers import AutoTokenizer
from tqdm import tqdm
tqdm.pandas()

# Cargar API key de Groq
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

RPD_LIMIT = 500

# Cliente de OpenAI apuntando a OpenRouter
client = OpenAI(
  api_key=OPENROUTER_API_KEY,
  base_url="https://openrouter.ai/api/v1",
)

# Selección de modelo
# MODEL = "meta-llama/llama-3.1-8b-instruct"
# base_model = "llama-3.1-8b"
# o
MODEL = "openai/gpt-oss-20b"
base_model = "gpt-oss-20b"

## Cargar dataset

In [ ]:
# conectar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Cargar datasets individuales por tarea
path = "/content/drive/MyDrive/tfg/corpusMentalRiskES/splits/LLM/"
path_tara5 = path + "tara5/"

df_anx = pd.read_csv(path + "anxiety_llm.csv", sep=",")
df_dep = pd.read_csv(path + "depression_llm.csv", sep=",")
df_ed = pd.read_csv(path + "ed_llm.csv", sep=",")
df_multi = pd.read_csv(path + "multiclass_llm.csv", sep=",")

df_anx_tara5 = pd.read_csv(path_tara5 + "anxiety_llm_tara5.csv")
df_dep_tara5 = pd.read_csv(path_tara5 + "depression_llm_tara5.csv")
df_ed_tara5  = pd.read_csv(path_tara5 + "ed_llm_tara5.csv")
df_multi_tara5 = pd.read_csv(path_tara5 + "multiclass_llm_tara5.csv")

# Seleccionar tarea de clasificacion: anx | dep | ed | multi
task = "multi" # <--- Cambiar aquí

# Asignar el dataframe correspondiente a la tarea seleccionada
if task == "anx":
  df = df_anx
  df_tara5 = df_anx_tara5
elif task == "dep":
  df = df_dep
  df_tara5 = df_dep_tara5
elif task == "ed":
  df = df_ed
  df_tara5 = df_ed_tara5
elif task == "multi":
  df = df_multi
  df_tara5 = df_multi_tara5

## Función para construir el prompt

Se añade el texto a clasificar a un prompt fijo que cambio según la tarea.

In [ ]:
def build_prompt(text, task):
  """ Construcción dinámica del prompt según tarea.
  Devuelve el texto completo que se enviará al modelo.
  """

  if task == "anx":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de ansiedad.

La ansiedad se asocia a una preocupación persistente y difícil de controlar, acompañada de nerviosismo, inquietud interna y una sensación continua de amenaza o anticipación negativa. Estas experiencias suelen generar malestar emocional sostenido y dificultad para relajarse.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de ansiedad
0: el texto no muestra indicios de ansiedad

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente: 0 o 1. No añadas texto.
"""
  elif task == "dep":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de depresión.

La depresión se asocia a un estado persistente de tristeza, apatía o vacío emocional, acompañado de una disminución del interés o la capacidad de disfrute en actividades habituales. Estas experiencias pueden incluir pensamientos negativos recurrentes, fatiga emocional y una percepción pesimista de uno mismo o del futuro.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de depresión
0: el texto no muestra indicios de depresión

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente: 0 o 1. No añadas texto.
"""
  elif task == "ed":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de un trastorno de la conducta alimentaria.

Los trastornos de la conducta alimentaria se asocian a una preocupación persistente por el peso corporal, la alimentación o la imagen corporal, que puede manifestarse mediante pensamientos rígidos o negativos sobre la comida, el cuerpo o el control del peso. Estas preocupaciones suelen generar malestar emocional, culpa, ansiedad en torno a la alimentación y conductas de evitación o control.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de un trastorno de la conducta alimentaria
0: el texto no muestra indicios de un trastorno de la conducta alimentaria

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente: 0 o 1. No añadas texto.
"""
  else:
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede reflejar indicios de uno de los siguientes estados psicológicos, o no reflejar ninguno de ellos.

Ansiedad: se asocia a una preocupación persistente y difícil de controlar, acompañada de nerviosismo, inquietud interna y una sensación continua de amenaza o anticipación negativa, generando malestar emocional y dificultad para relajarse.

Depresión: se asocia a un estado persistente de tristeza, apatía o vacío emocional, junto con una disminución del interés o la capacidad de disfrute en actividades habituales, pensamientos negativos recurrentes y una visión pesimista de uno mismo o del futuro.

Trastornos de la conducta alimentaria: se asocian a una preocupación persistente por el peso, la alimentación o la imagen corporal, que puede manifestarse mediante pensamientos rígidos o negativos sobre la comida o el cuerpo, así como malestar emocional, culpa o ansiedad en torno a la alimentación.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

0: el texto no muestra indicios de ninguno de los estados descritos
1: el texto muestra indicios de ansiedad
2: el texto muestra indicios de depresión
3: el texto muestra indicios de un trastorno de la conducta alimentaria

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente: 0, 1, 2 o 3. No añadas texto.
"""

## Calcular número de tokens

In [ ]:
HF_TOKEN = userdata.get("HF_TOKEN")

# Obtener el tokenizador correspondiente al modelo
tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct",
    token=HF_TOKEN
)
# tokenizer = AutoTokenizer.from_pretrained(
#     "openai/gpt-oss-20b"
# )

def total_tokens_dataset(df, task):
  """ Calcula el número total de tokens de entrada
  para un dataset completo, usando el tokenizador del modelo
  """
  total_tokens = 0

  for text in df["texto"]:
    prompt = build_prompt(text, task)
    tokens = len(tokenizer(prompt)["input_ids"])
    total_tokens += tokens

  return total_tokens


# Cálculo de tokens por dataset
total_anx = total_tokens_dataset(df_anx, "anx")
print("DATASET ANSIEDAD")
print("Tokens:", total_anx)
print("Media por texto:", total_anx / len(df_anx))

total_dep = total_tokens_dataset(df_dep, "dep")
print()
print("DATASET DEPRESIÓN")
print("Tokens:", total_dep)
print("Media por texto:", total_dep / len(df_dep))

total_ed = total_tokens_dataset(df_ed, "ed")
print()
print("DATASET ED")
print("Tokens:", total_ed)
print("Media por texto:", total_ed / len(df_ed))

total_multi = total_tokens_dataset(df_multi, "multi")
print()
print("DATASET MULTICLASE")
print("Tokens:", total_multi)
print("Media por texto:", total_multi / len(df_multi))

# Suma total de tokens necesarios para todo el experimento
print()
print(f"Total de tokens: {total_anx + total_dep + total_ed + total_multi}")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

DATASET ANSIEDAD
Tokens: 516451
Media por texto: 1032.902

DATASET DEPRESIÓN
Tokens: 438951
Media por texto: 879.6613226452906

DATASET ED
Tokens: 303037
Media por texto: 904.5880597014925

DATASET MULTICLASE
Tokens: 1486330
Media por texto: 1114.190404797601

Total de tokens: 2744769


## Clasificación zero-shot mediante modelos de OpenRouter

In [ ]:
def classify_text(text, task):
    """Función de clasificación usando OpenAI.
    Devuelve:
    - int (0,1,2,3) si hay predicción válida
    - None si el modelo bloquea o no devuelve texto
    """
    try:
        start = time.time()

        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "user", "content": build_prompt(text, task)}
            ],
            temperature=0
        )

        output_text = response.choices[0].message.content

        if not output_text:
            print("Resultado Vacio")
            return None

        match = re.search(r"[0-3]", output_text)
        return int(match.group()) if match else None

    except Exception as e:
        print("ERROR:", e)
        return None

In [ ]:
checkpoint_dir = "/content/drive/MyDrive/tfg/corpusMentalRiskES/checkpoints/"
output_dir = "/content/drive/MyDrive/tfg/corpusMentalRiskES/resultados/"

os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

def classify_with_daily_limit(df, task, rpd_limit=RPD_LIMIT, run_id=None):
  """
  Clasifica el dataset respetando el límite diario de la API.

  Lógica:
  - Carga el checkpoint si existe (reanuda desde donde se quedó)
  - Procesa hasta rpd_limit textos por ejecución
  - Guarda el checkpoint al finalizar la sesión del día
  - Cuando todos los textos están clasificados, guarda el resultado final

  Parámetros:
  - df: DataFrame con columna 'texto'
  - task: tarea ('anx', 'dep', 'ed', 'multi')
  - rpd_limit: máximo de llamadas a la API en esta ejecución
  """

  if run_id is None:
    checkpoint_path = checkpoint_dir + f"checkpoint_{base_model.replace('/', '_')}_{task}.csv"
  else:
    checkpoint_path = checkpoint_dir + f"tara5_{base_model.replace('/', '_')}_{task}_run{run_id}.csv"

  # --- Cargar checkpoint si existe ---
  if os.path.exists(checkpoint_path):
    df_ckpt = pd.read_csv(checkpoint_path)
    df["pred"] = df_ckpt["pred"]
    ya_procesados = int(df["pred"].notna().sum())
    print(f"✅ Checkpoint encontrado: {ya_procesados}/{len(df)} textos ya procesados.")
  else:
    df["pred"] = None
    ya_procesados = 0
    print(f"🆕 Sin checkpoint previo para {checkpoint_path}. Empezando desde cero.")

  pendientes = int(df["pred"].isna().sum())

  if pendientes == 0:
    print("🎉 Dataset ya completado. No hay nada que procesar.")
    return df

  llamadas_hoy = 0
  bloqueados = 0

  print(f"📋 Textos pendientes: {pendientes} | Límite de hoy: {rpd_limit}")
  print(f"📅 Días restantes estimados: {-(-pendientes // rpd_limit)}")  # ceil division

  # Barra de progreso sobre los textos que procesaremos hoy
  pbar = tqdm(total=min(pendientes, rpd_limit), desc="Progreso de hoy")

  for i, row in df.iterrows():

    # Parar si alcanzamos el límite diario
    if llamadas_hoy >= rpd_limit:
      print(f"\n🛑 Límite diario alcanzado ({rpd_limit} llamadas). Vuelve mañana.")
      break

    # Saltar filas ya procesadas
    if pd.notna(df.at[i, "pred"]):
      continue

    start_time = time.time()

    df.at[i, "pred"] = classify_text(row["texto"], task)
    llamadas_hoy += 1

    if df.at[i, "pred"] is None:
      bloqueados += 1

    pbar.update(1)
    pbar.set_postfix({"bloqueados": bloqueados, "hoy": llamadas_hoy})

    # Control de rate limit (25 RPM)
    elapsed = time.time() - start_time
    sleep_time = max(0, 2.5 - elapsed)
    time.sleep(sleep_time)

  pbar.close()

  # --- Guardar checkpoint ---
  df.to_csv(checkpoint_path, index=False)
  total_procesados = int(df["pred"].notna().sum())
  print(f"\n💾 Checkpoint guardado. Total procesados: {total_procesados}/{len(df)}")

  # --- Si el dataset está completo, guardar resultado final ---
  if df["pred"].isna().sum() == 0:
    output_path = output_dir + f"{base_model.replace('/', '_')}_{task}.csv"
    df.to_csv(output_path, index=False)
    print(f"🎉 Dataset completado. Resultado final guardado en:\n   {output_path}")

  return df, llamadas_hoy

In [ ]:
# Aplicar clasificación al dataset seleccionado
# Se almacena la predicción en una nueva columna "pred"
print(f"Iniciando sesión | Tarea: {task} | Modelo: {MODEL}")
df, _ = classify_with_daily_limit(df, task)

In [ ]:
df.to_csv(checkpoint_dir + f"checkpoint_{base_model.replace('/', '_')}_{task}.csv", index=False)
print("Checkpoint guardado manualmente")

Checkpoint guardado manualmente


### Comprobación de predicciones

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

# Reemplazar None por una clase inválida para que cuente como error
pred_labels = df["pred"].fillna(-1)

# Si es clasificación binaria
if task in ["anx", "dep", "ed"]:
    # Columna con la etiqueta real
    true_labels = df["bs"]
    f1 = f1_score(true_labels, pred_labels, average="macro")
else:
    # Columna con la etiqueta real
    true_labels = df["label_mc"]
    # Clasificación multiclase
    f1 = f1_score(true_labels, pred_labels, average="macro")

acc = accuracy_score(true_labels, pred_labels)

print("F1-score:", round(f1, 4))
print("Acc:", round(acc, 4))

F1-score: 0.7866
Acc: 0.7894


In [ ]:
# Número de bloqueos
num_blocked = df["pred"].isna().sum()
block_rate = num_blocked / len(df)

print("Bloqueos:", num_blocked)
print("Tasa de bloqueo:", round(block_rate, 4))

Bloqueos: 0
Tasa de bloqueo: 0.0


### Almacenar predicciones

In [ ]:
output_path = (
    f"/content/drive/MyDrive/tfg/corpusMentalRiskES/resultados/"
    f"{base_model}_{task}.csv"
)
df.to_csv(output_path, index=False)

## Clasificación múltiple para Tara@5

Para analizar la estabilidad de las predicciones, se aplicará TARa@5 sobre un subconjunto estratificado del dataset. Cada ejemplo se evalúa cinco veces con el mismo modelo, reutilizando la primera predicción y generando cuatro ejecuciones adicionales, lo que permite medir el grado de consistencia de las salidas.

In [ ]:
print(f"Iniciando TARa@5 para dataset: {task} | Modelo: {MODEL} | Textos: {len(df_tara5)}")

output_path = (
    f"/content/drive/MyDrive/tfg/corpusMentalRiskES/resultados/"
    f"{base_model}_{task}_tara5.csv"
)

# Aplicar la misma función de clasificación 4 veces más
# Se almacena cada predicción en una nueva columna "pred_run_{num_run}"
for run in range(1, 5):
    col_name = f"pred_run{run+1}"

    if col_name in df_tara5.columns:
        print(f"{col_name} ya existe, se omite")
        continue

    print(f"\nEjecutando pred_run{run+1}...")

    df_tara5[col_name] = df_tara5["texto"].progress_apply(lambda x: classify_text(x, task))

    # Guardado incremental
    df_tara5.to_csv(output_path, index=False)

    print(f"Run {run+1} guardada correctamente")

Iniciando TARa@5 para dataset: multi | Modelo: openai/gpt-oss-20b | Textos: 250

Ejecutando pred_run2...


 34%|███▎      | 84/250 [27:08<9:59:24, 216.65s/it]

Resultado Vacio


100%|██████████| 250/250 [55:26<00:00, 13.31s/it]


Run 2 guardada correctamente

Ejecutando pred_run3...


...

Run 4 guardada correctamente

Ejecutando pred_run5...


 32%|███▏      | 81/250 [09:51<34:04, 12.10s/it]

Resultado Vacio


100%|██████████| 250/250 [32:49<00:00,  7.88s/it]

Run 5 guardada correctamente
